In [1]:
!pip install pyspark


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("BigData Project Dataset Preview") \
    .getOrCreate()


# Load the small dataset
daily_df = spark.read.parquet(r"/content/drive/MyDrive/daily_data_fixed.parquet")

# Load the  dataset big
hourly_df = spark.read.parquet(r"/content/drive/MyDrive/hourly_data_fixed.parquet")

# Show basic info for daily data
print("=== Daily Data ===")
daily_df.show(5)  # Show first 5 rows
daily_df.printSchema()  # Show column names and types
print(f"Total rows: {daily_df.count()}")

# Show basic info for hourly data
print("\n=== Hourly Data (Large Dataset) ===")
hourly_df.show(5)
hourly_df.printSchema()
print(f"Total rows: {hourly_df.count()}")

# Stop Spark session
spark.stop()

=== Daily Data ===
+---------+-------------------+------------+------------------+------------------+-------------------+------------------------+------------------------+-------------------------+----------------+----------------+-----------------+-----------------+-----------------+--------+------------+-------------------+------------------+------------------+---------------------------+-----------------------+--------------------------+----+
|city_name|           datetime|weather_code|temperature_2m_max|temperature_2m_min|temperature_2m_mean|apparent_temperature_max|apparent_temperature_min|apparent_temperature_mean|         sunrise|          sunset|daylight_duration|sunshine_duration|precipitation_sum|rain_sum|snowfall_sum|precipitation_hours|wind_speed_10m_max|wind_gusts_10m_max|wind_direction_10m_dominant|shortwave_radiation_sum|et0_fao_evapotranspiration|year|
+---------+-------------------+------------+------------------+------------------+-------------------+-----------------

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# 1. Start Spark session
spark = SparkSession.builder \
    .appName("Daily Weather Classification - Spark") \
    .getOrCreate()

# Reduce log verbosity
spark.sparkContext.setLogLevel("WARN")

# 2. Load the daily dataset
df = spark.read.parquet(r"/content/drive/MyDrive/daily_data_fixed.parquet")

# 3. Select features (excluding string/timestamp columns)
feature_cols = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean',
    'daylight_duration', 'sunshine_duration', 'precipitation_sum', 'rain_sum', 'snowfall_sum',
    'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max',
    'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration'
]

# 4. Drop nulls in relevant columns
df_clean = df.select(['weather_code'] + feature_cols).dropna()

# 5. Encode target
indexer = StringIndexer(inputCol="weather_code", outputCol="label")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# 6. Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# 7. Split data
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Cache training data to speed up training
train_data.cache()

# 8–10. Train & Evaluate with different partition counts
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Repartition training only
train_repart = train_data.coalesce(2)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)  # keep test_data as is
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 2 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")




=== 2 Partitions ===
Training time: 141.23 seconds
Prediction time: 0.15 seconds
Test Accuracy: 0.6382


In [ ]:
train_repart = train_data.coalesce(4)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 4 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")


=== 4 Partitions ===
Training time: 108.52 seconds
Prediction time: 0.15 seconds
Test Accuracy: 0.6335


In [5]:
train_repart = train_data.coalesce(8)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 8 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")

=== 8 Partitions ===
Training time: 91.83 seconds
Prediction time: 0.14 seconds
Test Accuracy: 0.6382


In [6]:
train_repart = train_data.coalesce(32)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 32 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")

=== 32 Partitions ===
Training time: 95.92 seconds
Prediction time: 0.22 seconds
Test Accuracy: 0.6382


In [7]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# 1. Start Spark session
spark = SparkSession.builder \
    .appName("Hourly Weather Classification - Spark") \
    .getOrCreate()

# Reduce Spark log verbosity
spark.sparkContext.setLogLevel("WARN")

# 2. Load the hourly dataset
df = spark.read.parquet(r"/content/drive/MyDrive/hourly_data_fixed.parquet")

# 3. Choose a subset of numeric features (excluding soil/irradiance to reduce training time)
feature_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'apparent_temperature', 'precipitation', 'pressure_msl',
    'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
    'et0_fao_evapotranspiration'
]

# 4. Drop nulls in selected features + label
df_clean = df.select(['weather_code'] + feature_cols).dropna()

# 5. Encode label
indexer = StringIndexer(inputCol="weather_code", outputCol="label")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# 6. Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# 7. Train/Test split
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Cache training data to speed up training
train_data.cache()

# 8. Train Random Forest with timing
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)

start_train = time.time()
model = rf.fit(train_data)
end_train = time.time()

# 9. Predict with timing
start_pred = time.time()
predictions = model.transform(test_data)
end_pred = time.time()

# 10. Evaluate accuracy and F1-score
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1_score = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1_score:.4f}")

# Stop Spark
spark.stop()


Training time: 4219.60 seconds
Prediction time: 0.22 seconds
Test Accuracy: 0.9338
F1 Score: 0.9258


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# 1. Start Spark session
spark = SparkSession.builder \
    .appName("Hourly Weather Classification - Spark") \
    .getOrCreate()

# Reduce Spark log verbosity
spark.sparkContext.setLogLevel("WARN")

# 2. Load the hourly dataset
df = spark.read.parquet(r"/content/drive/MyDrive/hourly_data_fixed.parquet")

# 3. Choose a subset of numeric features (excluding soil/irradiance to reduce training time)
feature_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'apparent_temperature', 'precipitation', 'pressure_msl',
    'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
    'et0_fao_evapotranspiration'
]

# 4. Drop nulls in selected features + label
df_clean = df.select(['weather_code'] + feature_cols).dropna()

# 5. Encode label
indexer = StringIndexer(inputCol="weather_code", outputCol="label")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# 6. Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# 7. Train/Test split
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Cache training data to speed up training
train_data.cache()

# 8. Train Random Forest with timing

# 8–10. Train & Evaluate with different partition counts
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Repartition training only
train_repart = train_data.coalesce(2)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)  # keep test_data as is
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 2 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")




=== 2 Partitions ===
Training time: 4125.04 seconds
Prediction time: 0.12 seconds
Test Accuracy: 0.9352


In [8]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# 1. Start Spark session
spark = SparkSession.builder \
    .appName("Hourly Weather Classification - Spark") \
    .getOrCreate()

# Reduce Spark log verbosity
spark.sparkContext.setLogLevel("WARN")

# 2. Load the hourly dataset
df = spark.read.parquet(r"/content/drive/MyDrive/hourly_data_fixed.parquet")

# 3. Choose a subset of numeric features (excluding soil/irradiance to reduce training time)
feature_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'apparent_temperature', 'precipitation', 'pressure_msl',
    'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
    'et0_fao_evapotranspiration'
]

# 4. Drop nulls in selected features + label
df_clean = df.select(['weather_code'] + feature_cols).dropna()

# 5. Encode label
indexer = StringIndexer(inputCol="weather_code", outputCol="label")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# 6. Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# 7. Train/Test split
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Cache training data to speed up training
train_data.cache()

# 8. Train Random Forest with timing

# 8–10. Train & Evaluate with different partition counts
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Repartition training only
train_repart = train_data.coalesce(8)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)  # keep test_data as is
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 8 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")




=== 8 Partitions ===
Training time: 3311.59 seconds
Prediction time: 0.06 seconds
Test Accuracy: 0.9433


In [9]:
spark.stop()

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# 1. Start Spark session
spark = SparkSession.builder \
    .appName("Hourly Weather Classification - Spark") \
    .getOrCreate()

# Reduce Spark log verbosity
spark.sparkContext.setLogLevel("WARN")

# 2. Load the hourly dataset
df = spark.read.parquet(r"/content/drive/MyDrive/hourly_data_fixed.parquet")

# 3. Choose a subset of numeric features (excluding soil/irradiance to reduce training time)
feature_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'apparent_temperature', 'precipitation', 'pressure_msl',
    'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
    'et0_fao_evapotranspiration'
]

# 4. Drop nulls in selected features + label
df_clean = df.select(['weather_code'] + feature_cols).dropna()

# 5. Encode label
indexer = StringIndexer(inputCol="weather_code", outputCol="label")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# 6. Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# 7. Train/Test split
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Cache training data to speed up training
train_data.cache()

# 8. Train Random Forest with timing

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Repartition training only
train_repart = train_data.coalesce(32)

start_train = time.time()
model = rf.fit(train_repart)
end_train = time.time()

start_pred = time.time()
predictions = model.transform(test_data)  # keep test_data as is
end_pred = time.time()

accuracy = evaluator.evaluate(predictions)

print("=== 32 Partitions ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")




=== 32 Partitions ===
Training time: 4793.95 seconds
Prediction time: 0.18 seconds
Test Accuracy: 0.9235
